## 마스킹

In [1]:
import cv2
import numpy as np
import pathlib
import os
from tqdm import tqdm

# =========================================================
# [설정 영역]
# =========================================================

# 1. 입력 경로 (오버레이/주석/빨간마스크가 포함된 이미지가 있는 폴더)
INPUT_ROOT = pathlib.Path('/home/msko021220/project/Dataset_BUSI_with_GT/Final_Masked')

# 2. 출력 경로 (생성된 마스크가 저장될 폴더)
OUTPUT_ROOT = pathlib.Path('/home/msko021220/project/Dataset_BUSI_with_GT/Final_Generated_Masks')

# 3. 처리할 하위 폴더 리스트
TARGET_FOLDERS = ['benign', 'malignant']

# =========================================================

def detect_overlay_mask(img_path: pathlib.Path) -> np.ndarray:
    """
    사용자가 제공한 로직: 빨간색, 노란색, 흰색 오버레이를 감지하여 마스크 반환
    """
    # 1. 이미지 로드 (BGR 컬러 모드)
    # pathlib.Path 객체를 문자열로 변환하여 cv2.imread에 전달
    bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)

    if bgr is None:
        print(f"⚠️ 이미지 로드 실패: {img_path}")
        return None

    # 2. BGR -> HSV 색상 공간 변환
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

    # 3. 색상 범위 정의

    # [Red] 빨간색 (0도 근처 & 180도 근처)
    mask_r1 = cv2.inRange(hsv, (0, 70, 70), (10, 255, 255))
    mask_r2 = cv2.inRange(hsv, (170, 70, 70), (180, 255, 255))

    # [Yellow] 노란색
    mask_y = cv2.inRange(hsv, (20, 80, 80), (35, 255, 255))

    # [White] 흰색 텍스트/십자가 (채도 낮고 명도 높음)
    mask_w = cv2.inRange(hsv, (0, 0, 200), (180, 50, 255))

    # 4. 모든 마스크 합치기 (Bitwise OR)
    mask = mask_r1 | mask_r2 | mask_y | mask_w

    # 5. 팽창 (Dilation) 적용
    # 3x3 커널로 마스크 영역을 살짝 넓혀서 경계선을 확실히 덮음
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)

    return mask

def run_processing():
    print(f"🚀 마스킹 작업을 시작합니다...")
    print(f"📂 입력 경로: {INPUT_ROOT}")
    print(f"💾 출력 경로: {OUTPUT_ROOT}\n")

    for folder in TARGET_FOLDERS:
        # 입력/출력 서브 폴더 경로 설정
        input_dir = INPUT_ROOT / folder
        output_dir = OUTPUT_ROOT / folder

        # 출력 폴더가 없으면 생성
        output_dir.mkdir(parents=True, exist_ok=True)

        # 입력 폴더 확인
        if not input_dir.exists():
            print(f"⚠️ 폴더 없음: {input_dir}")
            continue

        # .png 파일 리스트 가져오기
        image_files = sorted(list(input_dir.glob("*.png")))
        print(f"Processing [{folder}] - 총 {len(image_files)}장")

        for img_path in tqdm(image_files):
            try:
                # 1. 마스크 생성 함수 호출
                mask = detect_overlay_mask(img_path)

                if mask is None:
                    continue

                # 2. 결과 저장
                # 파일명은 원본과 동일하게 하거나, 구분하고 싶다면 아래처럼 변경 가능
                # save_name = img_path.stem + "_mask.png"
                save_name = img_path.name
                save_path = output_dir / save_name

                cv2.imwrite(str(save_path), mask)

            except Exception as e:
                print(f"❌ Error ({img_path.name}): {e}")
                continue

    print(f"\n✅ 모든 작업 완료! 결과물을 확인해주세요: {OUTPUT_ROOT}")

# 실행
if __name__ == "__main__":
    run_processing()

🚀 마스킹 작업을 시작합니다...
📂 입력 경로: /home/msko021220/project/Dataset_BUSI_with_GT/Final_Masked
💾 출력 경로: /home/msko021220/project/Dataset_BUSI_with_GT/Final_Generated_Masks

Processing [benign] - 총 437장


  0%|          | 0/437 [00:00<?, ?it/s]

100%|██████████| 437/437 [00:34<00:00, 12.64it/s]


Processing [malignant] - 총 210장


100%|██████████| 210/210 [00:14<00:00, 14.13it/s]


✅ 모든 작업 완료! 결과물을 확인해주세요: /home/msko021220/project/Dataset_BUSI_with_GT/Final_Generated_Masks


In [19]:
import cv2
import numpy as np
import os
import glob
from tqdm import tqdm

# =========================================================
# [설정 영역]
# =========================================================

# 1. 경로 설정 (사용자 지정 경로)
BASE_ROOT = '/home/msko021220/project/Dataset_BUSI_with_GT'
INPUT_ROOT = os.path.join(BASE_ROOT, 'Final_Masked')
OUTPUT_ROOT = os.path.join(BASE_ROOT, 'Final_Red_Masks_Conservative') # 폴더명 변경 (보수적 모드)
TARGET_FOLDERS = ['benign', 'malignant']

# [핵심 튜닝 파라미터] - 정상 조직 보호를 위해 값을 엄격하게 조정함

# 1. 절대 밝기 커트라인 (0~255)
# 이 값보다 어두운 픽셀은 절대 잡지 않음. 정상 조직(회색~연한 흰색)을 피하기 위함.
ABSOLUTE_MIN_BRIGHTNESS = 215

# 2. 상대 밝기(Top-Hat) 민감도
# 배경보다 얼마나 더 밝아야 잡을 것인가? (높을수록 엄격함)
MARKER_THRESHOLD = 70

# 3. 감지 커널 크기
# 값을 줄여서 '큰 덩어리(조직)'는 무시하고 '작은 점/선'만 타겟팅
TOPHAT_KERNEL_SIZE = (15, 15)

# =========================================================

def detect_strict_white_markers(img_bgr):
    """
    정상 조직 보호를 위한 이중 필터링 적용 마커 감지
    """
    if img_bgr is None: return None
    
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    # [필터 1] 절대 밝기 필터링 (Absolute Threshold)
    # 픽셀 자체가 아주 밝은 흰색이 아니면 무시 (정상 조직 배제)
    _, abs_mask = cv2.threshold(gray, ABSOLUTE_MIN_BRIGHTNESS, 255, cv2.THRESH_BINARY)
    
    # [필터 2] 형태학적 Top-Hat 필터링 (Relative Contrast)
    # 배경보다 국소적으로 톡 튀어나온 구조물 감지
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, TOPHAT_KERNEL_SIZE)
    tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel)
    _, rel_mask = cv2.threshold(tophat, MARKER_THRESHOLD, 255, cv2.THRESH_BINARY)
    
    # [교집합] 두 조건을 모두 만족해야 진짜 마커로 인정
    # "원래 아주 밝으면서" AND "주변보다 튀어나와 있어야 함"
    final_mask = cv2.bitwise_and(abs_mask, rel_mask)
    
    # 노이즈 제거 (아주 미세한 1px 점들은 무시)
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, np.ones((2,2), np.uint8))
    
    # 점선 연결을 위한 살짝 팽창
    # 너무 많이 넓히면 정상 조직 침범하므로 적당히(3x3) 조절
    final_mask = cv2.dilate(final_mask, np.ones((3,3), np.uint8), iterations=1)
    
    return final_mask

def run_safe_masking():
    print(f"🚀 [안전 모드] 정상 조직 보호 마스킹 시작...")
    print(f"📂 입력 경로: {INPUT_ROOT}")
    print(f"💾 출력 경로: {OUTPUT_ROOT}")
    print(f"⚙️ 설정: 밝기 > {ABSOLUTE_MIN_BRIGHTNESS} AND 대비 > {MARKER_THRESHOLD}\n")
    
    if not os.path.exists(INPUT_ROOT):
        print(f"❌ 입력 경로 없음: {INPUT_ROOT}")
        return

    for folder in TARGET_FOLDERS:
        src_dir = os.path.join(INPUT_ROOT, folder)
        dst_dir = os.path.join(OUTPUT_ROOT, folder)
        os.makedirs(dst_dir, exist_ok=True)

        if not os.path.exists(src_dir): continue

        image_files = sorted(glob.glob(os.path.join(src_dir, "*.png")))
        print(f"Processing [{folder}] - {len(image_files)}장")

        for img_path in tqdm(image_files):
            filename = os.path.basename(img_path)
            
            try:
                img_bgr = cv2.imread(img_path)
                if img_bgr is None: continue
                
                # 엄격한 마커 감지
                white_marker_mask = detect_strict_white_markers(img_bgr)
                
                # 원본에 빨간색 칠하기
                img_bgr[white_marker_mask > 0] = [0, 0, 255]
                
                cv2.imwrite(os.path.join(dst_dir, filename), img_bgr)

            except Exception as e:
                print(f"Err ({filename}): {e}")

    print(f"\n✅ 완료! 결과물 위치: {OUTPUT_ROOT}")

if __name__ == "__main__":
    run_safe_masking()

🚀 [안전 모드] 정상 조직 보호 마스킹 시작...
📂 입력 경로: /home/msko021220/project/Dataset_BUSI_with_GT/Final_Masked
💾 출력 경로: /home/msko021220/project/Dataset_BUSI_with_GT/Final_Red_Masks_Conservative
⚙️ 설정: 밝기 > 215 AND 대비 > 70

Processing [benign] - 437장


100%|██████████| 437/437 [00:09<00:00, 44.83it/s]


Processing [malignant] - 210장


100%|██████████| 210/210 [00:04<00:00, 48.65it/s]


✅ 완료! 결과물 위치: /home/msko021220/project/Dataset_BUSI_with_GT/Final_Red_Masks_Conservative


In [14]:
import os
from accelerate.utils import write_basic_config

# 1. 버전이 맞지 않는 기존 스크립트 삭제 (필수)
if os.path.exists("train_dreambooth_lora.py"):
    os.remove("train_dreambooth_lora.py")
    print("🗑️ 호환되지 않는 스크립트를 삭제했습니다.")

# 2. 현재 설치된 diffusers 버전(0.36.0)에 맞는 스크립트 다운로드
print("⬇️ 호환되는 버전(v0.36.0)의 학습 스크립트를 다운로드합니다...")
!wget https://raw.githubusercontent.com/huggingface/diffusers/v0.36.0/examples/dreambooth/train_dreambooth_lora.py

# 3. accelerate 설정 (혹시 풀렸을 경우를 대비)
write_basic_config()

# === [설정 영역] ===
DATA_DIR = "/home/msko021220/project/Dataset_BUSI_with_GT/normal_fixed"
OUTPUT_DIR = "/home/msko021220/project/Dataset_BUSI_with_GT/LoRA"
# =================

print(f"\n🚀 학습을 다시 시작합니다!")
print(f"데이터 경로: {DATA_DIR}")
print(f"저장 경로: {OUTPUT_DIR}")

# 4. 학습 실행
# 주의: 'runwayml/stable-diffusion-v1-5' 대신 'stable-diffusion-v1-5/stable-diffusion-v1-5'를 사용
# ★ 핵심 수정: accelerate 실행 파일의 절대 경로를 사용하여 PATH 문제 해결
!/home/msko021220/.local/bin/accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --instance_data_dir="{DATA_DIR}" \
  --output_dir="{OUTPUT_DIR}" \
  --instance_prompt="a photo of sks ultrasound" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --checkpointing_steps=500 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1000 \
  --seed="0"

🗑️ 호환되지 않는 스크립트를 삭제했습니다.
⬇️ 호환되는 버전(v0.36.0)의 학습 스크립트를 다운로드합니다...
--2026-01-14 11:33:17--  https://raw.githubusercontent.com/huggingface/diffusers/v0.36.0/examples/dreambooth/train_dreambooth_lora.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 59186 (58K) [text/plain]
Saving to: ‘train_dreambooth_lora.py’

train_dreambooth_lo 100%[===================>]  57.80K  --.-KB/s    in 0.002s  

2026-01-14 11:33:17 (25.2 MB/s) - ‘train_dreambooth_lora.py’ saved [59186/59186]

Configuration already exists at /home/msko021220/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.

🚀 학습을 다시 시작합니다!
데이터 경로: /home/msko021220/project/Dataset_BUSI_with_GT/normal_fixed
저장 경로: /home/msko021

In [1]:
import torch
from diffusers import AutoPipelineForInpainting
from PIL import Image, ImageFilter
import os
import glob
from tqdm import tqdm
import numpy as np
import cv2

# =========================================================
# [설정 영역]
# =========================================================

# 1. 빨간색 마스크처리가 완료된 이미지가 있는 경로
dataset_root_path = '/home/msko021220/project/Dataset_BUSI_with_GT/Final_Red_Masks_Conservative'

# 2. 결과물을 저장할 경로
output_root_path = '/home/msko021220/project/Dataset_BUSI_with_GT/Inpainting_Results_Natural'

# 3. 학습한 LoRA 경로
lora_model_path = "/home/msko021220/project/Dataset_BUSI_with_GT/LoRA"

# 4. [모델 변경] Stable Diffusion v1.5 Inpainting 모델 ID
MODEL_ID = "runwayml/stable-diffusion-inpainting"

# [튜닝 파라미터]
# 텍스트/네모난 마킹 흔적을 지우기 위해 Dilation과 Blur를 넉넉하게 유지
MASK_DILATION = 20
MASK_BLUR_RADIUS = 15
INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5     # SD 1.5 표준 권장값

positive_prompt = (
    "medical ultrasound scan, healthy breast tissue, organic texture, "
    "black and white, grainy speckle noise, high quality, anatomical correctness"
)
if lora_model_path and os.path.exists(lora_model_path):
    positive_prompt = "a photo of sks ultrasound, " + positive_prompt

negative_prompt = (
    "(blurred text artifacts:1.4), (square patch:1.3), (inpainting seams:1.3), "
    "(text:1.5), (signature:1.5), (watermark:1.5), (label:1.3), (alphabet:1.3), "
    "(digits:1.3), (date:1.3), (logo:1.3), (ruler:1.3), (measurement:1.3), "
    "tumor, cyst, mass, lesion, color, face, human, person, artifacts, blur, low quality, distortion"
)

target_folders = ['benign', 'malignant']

# =========================================================

def extract_red_mask_robust(image_pil):
    """HSV 색상 공간을 사용하여 모든 종류의 빨간색 영역을 강력하게 추출"""
    img_rgb = np.array(image_pil)
    img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

    # 빨간색 범위 1 (Hue 0~10)
    lower_red1 = np.array([0, 50, 50])
    upper_red1 = np.array([10, 255, 255])
    # 빨간색 범위 2 (Hue 170~180)
    lower_red2 = np.array([170, 50, 50])
    upper_red2 = np.array([180, 255, 255])

    mask1 = cv2.inRange(img_hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(img_hsv, lower_red2, upper_red2)

    final_mask = mask1 | mask2
    return Image.fromarray(final_mask, mode='L')

def run_natural_inpainting():
    print(f"⏳ 모델 로드 중... ({MODEL_ID})")
    if not torch.cuda.is_available():
        print("❌ GPU가 필요합니다.")
        return

    # 모델 ID를 명시적으로 지정하여 로드
    pipe = AutoPipelineForInpainting.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        safety_checker=None
    ).to("cuda")

    if lora_model_path and os.path.exists(lora_model_path):
        print(f"🧠 LoRA 적용: {lora_model_path}")
        pipe.load_lora_weights(lora_model_path)

    if not os.path.exists(output_root_path): os.makedirs(output_root_path)

    for folder_name in target_folders:
        current_dir = os.path.join(dataset_root_path, folder_name)
        save_dir = os.path.join(output_root_path, folder_name)
        os.makedirs(save_dir, exist_ok=True)

        if not os.path.exists(current_dir): continue

        image_files = sorted(glob.glob(os.path.join(current_dir, "*.png")))
        print(f"\n🚀 Processing [{folder_name}] - {len(image_files)} images (Natural Mode)...")

        for img_path in tqdm(image_files):
            filename = os.path.basename(img_path)

            try:
                # 1. 이미지 로드
                image = Image.open(img_path).convert("RGB")
                original_size = image.size

                # 2. 빨간색 마스크 추출
                mask_image = extract_red_mask_robust(image)
                if not np.any(np.array(mask_image) > 0): continue

                # 3. 마스크 전처리 (자연스러운 텍스트 제거를 위한 Dilation & Blur)
                mask_array = np.array(mask_image)
                kernel = np.ones((MASK_DILATION, MASK_DILATION), np.uint8)
                dilated_mask = cv2.dilate(mask_array, kernel, iterations=1)

                mask_image_pil = Image.fromarray(dilated_mask)
                mask_image_blurred = mask_image_pil.filter(ImageFilter.GaussianBlur(radius=MASK_BLUR_RADIUS))

                # 4. 리사이즈 및 생성
                image_input = image.resize((512, 512))
                mask_input = mask_image_blurred.resize((512, 512))

                result = pipe(
                    prompt=positive_prompt,
                    negative_prompt=negative_prompt,
                    image=image_input,
                    mask_image=mask_input,
                    height=512, width=512,
                    num_inference_steps=INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE, # SD 1.5 권장값 7.5 적용
                    cross_attention_kwargs={"scale": 0.6} if lora_model_path else {}
                ).images[0]

                # 5. 저장
                result = result.resize(original_size)
                result.save(os.path.join(save_dir, filename))

            except Exception as e:
                print(f"Err ({filename}): {e}")
                continue

    print(f"\n✅ 자연스러운 변환 완료! 결과물 위치: {output_root_path}")

run_natural_inpainting()

⏳ 모델 로드 중... (runwayml/stable-diffusion-inpainting)


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

 34%|███▎      | 147/437 [11:14<21:58,  4.55s/it]

  0%|          | 0/50 [00:00<?, ?it/s]

 34%|███▎      | 147/437 [11:17<22:16,  4.61s/it]


KeyboardInterrupt: 